In [ ]:
import os
import sys
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from argparse import Namespace
from torch.utils.data import DataLoader, Subset
import kagglehub
import numpy as np

In [ ]:
import os
import sys

# -------------------------------
# 1. Download COCO dataset
# -------------------------------
import kagglehub

coco_path = kagglehub.dataset_download('awsaf49/coco-2017-dataset')
coco_data_root = os.path.join(coco_path, 'coco2017')

print("COCO dataset path:", coco_data_root)

# -------------------------------
# 2. Clone DETR repo (if not exists)
# -------------------------------
if not os.path.exists('detr'):
    os.system('git clone https://github.com/facebookresearch/detr.git')

# Add DETR to Python path
sys.path.insert(0, os.path.abspath('detr'))

# -------------------------------
# 3. Patch DETR to return hidden states (hs)
# -------------------------------
detr_py_path = 'detr/models/detr.py'

with open(detr_py_path, 'r') as f:
    code = f.read()

target_line = "return {'pred_logits': outputs_class[-1], 'pred_boxes': outputs_coord[-1]}"

if target_line in code and "'hs': hs[-1]" not in code:
    print("Patching DETR to return hidden states...")
    
    patched_line = "return {'pred_logits': outputs_class[-1], 'pred_boxes': outputs_coord[-1], 'hs': hs[-1]}"
    code = code.replace(target_line, patched_line)
    
    with open(detr_py_path, 'w') as f:
        f.write(code)
else:
    print("DETR already patched or target line not found.")

# -------------------------------
# 4. Patch box_ops to remove assertions
# -------------------------------
box_ops_path = 'detr/util/box_ops.py'

with open(box_ops_path, 'r') as f:
    content = f.read()

content = content.replace(
    'assert (boxes1[:, 2:] >= boxes1[:, :2]).all()',
    '# assertion removed'
)
content = content.replace(
    'assert (boxes2[:, 2:] >= boxes2[:, :2]).all()',
    '# assertion removed'
)

with open(box_ops_path, 'w') as f:
    f.write(content)

print("box_ops patched (assertions removed).")

# -------------------------------
# 5. Test import
# -------------------------------
try:
    from models.detr import DETR
    print("DETR import successful ✅")
except Exception as e:
    print("Import failed ❌:", e)

In [ ]:
import importlib

# Patch detr.py to include 'hs' in the output dictionary
detr_py_path = 'detr/models/detr.py'
with open(detr_py_path, 'r') as f: code = f.read()

# Target the actual assignment used in the Facebook repo
target_str = "out = {'pred_logits': outputs_class[-1], 'pred_boxes': outputs_coord[-1]}"
replacement_str = "out = {'pred_logits': outputs_class[-1], 'pred_boxes': outputs_coord[-1], 'hs': hs[-1]}"

if target_str in code:
    code = code.replace(target_str, replacement_str)
    with open(detr_py_path, 'w') as f: f.write(code)
    print("✅ detr.py patched successfully.")

# CRITICAL: Reload the module so the 'build' function sees the new code
import models.detr
importlib.reload(models.detr)
from models.detr import build

In [ ]:
class SOTATernary(nn.Module):
    def __init__(self, inn, out, bias=True, layer_name="unknown"):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out, inn))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        self.layer_name = layer_name
        # Adaptive Initialization
        d_init = 0.3 if any(x in layer_name for x in ["attn", "v_proj", "out_proj"]) else 0.6
        self.s_pos, self.s_neg, self.delta = nn.Parameter(torch.tensor(0.1)), nn.Parameter(torch.tensor(0.1)), nn.Parameter(torch.tensor(d_init))
        self.register_buffer('qweight', torch.zeros(out, inn, dtype=torch.int8))
        self.bias = nn.Parameter(torch.zeros(out)) if bias else None

    def forward(self, x):
        if self.training:
            mu, std = self.weight.mean(), self.weight.std() + 1e-5
            w_norm = (self.weight - mu) / std
            d = torch.abs(self.delta)
            temp = 10.0 
            mask_pos_soft = torch.sigmoid((w_norm - d) * temp)
            mask_neg_soft = torch.sigmoid((-w_norm - d) * temp)
            mask_pos_hard = (w_norm > d).float()
            mask_neg_hard = (w_norm < -d).float()
            mask_pos = mask_pos_hard + (mask_pos_soft - mask_pos_soft.detach())
            mask_neg = mask_neg_hard + (mask_neg_soft - mask_neg_soft.detach())
            w_quant = (mask_pos * torch.abs(self.s_pos)) - (mask_neg * torch.abs(self.s_neg))
            ewe = w_norm + (w_quant - w_norm).detach()
            return F.linear(x, ewe, self.bias)
        ewe = ((self.qweight == 1).float() * self.s_pos) - ((self.qweight == -1).float() * self.s_neg)
        return F.linear(x, ewe, self.bias)

In [ ]:
class INT8Linear(nn.Module):
    def __init__(self, inn, out, bias=True, layer_name="unknown"):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out, inn))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        self.bias = nn.Parameter(torch.zeros(out)) if bias else None

    def forward(self, x):
        scale = self.weight.abs().max().detach() / 127
        w_int8 = torch.clamp(torch.round(self.weight / (scale + 1e-8)), -127, 127)
        return F.linear(x, scale * (w_int8 + (self.weight/scale - (self.weight/scale).detach())), self.bias)


In [ ]:
def replace_with_ultra_low_bit(mod, name_prefix=""):
    for name, module in list(mod.named_children()):
        full_name = f"{name_prefix}.{name}" if name_prefix else name
        if isinstance(module, nn.Linear):
            # CASE: INT8 for Heads and QK (High precision shell)
            if any(h in full_name for h in ["class_embed", "bbox_embed", "q_proj", "k_proj", "in_proj"]):
                new = INT8Linear(module.in_features, module.out_features, bias=module.bias is not None, layer_name=full_name)
            # CASE: Ternary for Backbone/FFN (Efficient core)
            else:
                new = SOTATernary(module.in_features, module.out_features, bias=module.bias is not None, layer_name=full_name)
            new.weight.data.copy_(module.weight.data)
            if module.bias is not None: new.bias.data.copy_(module.bias.data)
            setattr(mod, name, new)
        else: replace_with_ultra_low_bit(module, full_name)
    return mod

In [ ]:
def get_args(pre_norm=True):
    return Namespace(dataset_file="coco", backbone="resnet50", num_queries=100, aux_loss=True, 
                     device="cuda", hidden_dim=256, dropout=0.1, nheads=8, enc_layers=6, 
                     dec_layers=6, dim_feedforward=2048, position_embedding="sine", 
                     dilation=False, normalize_before=False, lr_backbone=1e-5, masks=False, 
                     pre_norm=pre_norm, set_cost_class=1, set_cost_bbox=5, set_cost_giou=2, 
                     bbox_loss_coef=5, giou_loss_coef=2, eos_coef=0.1)

# Initialize models
teacher, _, _ = build(get_args(pre_norm=False))
teacher_ckpt = torch.hub.load_state_dict_from_url('https://dl.fbaipublicfiles.com/detr/detr-r50-e632da11.pth', map_location='cpu')
teacher.load_state_dict(teacher_ckpt['model']); teacher.to("cuda").eval()

In [ ]:
student, criterion, _ = build(get_args(pre_norm=True))
student = replace_with_ultra_low_bit(student); student.to("cuda")

In [ ]:
# Grouping parameters explicitly by name
params_rest = []
params_backbone = []
params_scales = []
params_deltas = []

for n, p in student.named_parameters():
    if not p.requires_grad: continue
    if "backbone" in n:
        params_backbone.append(p)
    elif "s_pos" in n or "s_neg" in n:
        params_scales.append(p)
    elif "delta" in n:
        params_deltas.append(p)
    else:
        params_rest.append(p)

optimizer = torch.optim.AdamW([
    {"params": params_rest,     "lr": 1e-4}, # Transformer & Heads
    {"params": params_backbone, "lr": 1e-5}, # ResNet
    {"params": params_scales,   "lr": 1e-4}, # Scale factors (fast)
    {"params": params_deltas,   "lr": 2e-6}  # Thresholds (slow & stable)
], weight_decay=1e-4)

In [ ]:
from datasets import build_dataset

In [ ]:
ds_args = Namespace(dataset_file='coco', coco_path=coco_data_root, masks=False)
dataset_train = Subset(build_dataset(image_set='train', args=ds_args), range(10000))
data_loader = DataLoader(dataset_train, batch_size=2, shuffle=True, collate_fn=lambda b: tuple(zip(*b)))
scaler = torch.cuda.amp.GradScaler()

In [ ]:
metric_history = []

In [ ]:
def run_research_analysis(model, epoch):
    """Calculates Sparsity and Delta evolution for the paper"""
    results = {"attn_delta": [], "ffn_delta": [], "attn_sparsity": [], "ffn_sparsity": []}
    with torch.no_grad():
        for name, m in model.named_modules():
            if isinstance(m, SOTATernary):
                mu, std = m.weight.mean(), m.weight.std() + 1e-5
                w_norm = (m.weight - mu) / std
                d = torch.abs(m.delta)
                sparsity = ((w_norm <= d) & (w_norm >= -d)).float().mean().item()
                if any(x in name for x in ["attn", "v_proj"]):
                    results["attn_delta"].append(d.item())
                    results["attn_sparsity"].append(sparsity)
                else:
                    results["ffn_delta"].append(d.item())
                    results["ffn_sparsity"].append(sparsity)
    
    print(f"\n📊 --- RESEARCH ANALYSIS EPOCH {epoch} ---")
    print(f"Avg Δ (Attention): {np.mean(results['attn_delta']):.4f} | Sparsity: {np.mean(results['attn_sparsity']):.2%}")
    print(f"Avg Δ (FFN):       {np.mean(results['ffn_delta']):.4f} | Sparsity: {np.mean(results['ffn_sparsity']):.2%}")
    return {k: float(np.mean(v)) for k, v in results.items()}

In [ ]:
metric_history = []
print("🔥 Starting SOTA Hybrid 10k Trial with Feature Distillation...")

for epoch in range(1, 21):
    student.train()
    epoch_loss = 0
    
    for i, (samples, targets) in enumerate(data_loader):
        # 1. Prepare Data
        samples = [s.to("cuda") for s in samples]
        targets = [{k: v.to("cuda") for k, v in t.items()} for t in targets]
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            # 2. Forward Passes
            out_s = student(samples)
            with torch.no_grad(): 
                out_t = teacher(samples)
            
            # 3. Feature Distillation (Matching Teacher's Internal State)
            dist_f = F.mse_loss(out_s['hs'], out_t['hs'])
            dist_c = F.kl_div(
                F.log_softmax(out_s['pred_logits']/2.0, -1),
                F.softmax(out_t['pred_logits']/2.0, -1),
                reduction='batchmean'
            ) * 4.0 
            # 4. Standard DETR Loss
            # Sanitize boxes for the Hungarian Matcher
            out_s['pred_boxes'] = torch.nan_to_num(out_s['pred_boxes'], nan=0.5).clamp(0, 1)
            loss_dict = criterion(out_s, targets)
            weight_dict = criterion.weight_dict
            base_l = sum(loss_dict[k] * weight_dict[k] for k in loss_dict.keys() if k in weight_dict)
            
            # 5. Combined Total Loss (Weighted Distillation)
            total_loss = base_l + (10.0 * dist_f)+dist_c 

        # 6. Backward and Optimize
        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(student.parameters(), 0.1) # Strict clipping for 1.58-bit
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += total_loss.item()

        # --- PRINT LOSS PER 100 STEWS ---
        if (i + 1) % 100 == 0:
            print(f"Epoch {epoch} | Iter {i+1} | Loss: {total_loss.item():.4f} | Distill_F: {dist_f.item():.6f}")

    # --- END OF EPOCH CALCULATION ---
    avg_loss = epoch_loss / len(data_loader)
    print(f"✅ Epoch {epoch} Complete | Avg Loss: {avg_loss:.4f}")

    # --- BIENNIAL ANALYSIS & SAVING (EVERY 2 EPOCHS) ---
    if epoch % 2 == 0:
        # Run research analysis (calculates sparsity/delta internally)
        stats = run_research_analysis(student, epoch)
        stats['avg_loss'] = avg_loss
        metric_history.append(stats)
        
        # Save checkpoint including the research metrics
        save_name = f"detr_sota_hybrid_10k_epoch{epoch}.pth"
        torch.save({
            'epoch': epoch,
            'model_state_dict': student.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'research_metrics': metric_history
        }, save_name)
        print(f"💾 Checkpoint and Sparsity Analysis saved to {save_name}")

print("🏁 10k Trial Complete.")